In [2]:
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("TensorFlow:", tf.__version__)
print("Num GPUs:", len(tf.config.list_physical_devices("GPU")))

TensorFlow: 2.20.0
Num GPUs: 2


In [3]:
DATASET_PATH = Path(
    "/kaggle/input/datasets/satishchandra9618/face-mask-data/Face_mask_ds"
)

MASK_DIR = DATASET_PATH / "Mask"
NO_MASK_DIR = DATASET_PATH / "No_mask"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

mask_files = sorted(
    p for p in MASK_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
)

no_mask_files = sorted(
    p for p in NO_MASK_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
)

print("Dataset path:", DATASET_PATH)
print()
print("Mask images   :", len(mask_files))
print("No-mask images:", len(no_mask_files))
print("Total images  :", len(mask_files) + len(no_mask_files))

Dataset path: /kaggle/input/datasets/satishchandra9618/face-mask-data/Face_mask_ds

Mask images   : 18197
No-mask images: 17513
Total images  : 35710


In [4]:
# Combine all image paths
all_files = mask_files + no_mask_files

# Labels:
# Mask    = 1
# No_mask = 0
all_labels = (
    [1] * len(mask_files) +
    [0] * len(no_mask_files)
)

print("Total images:", len(all_files))
print("Total labels:", len(all_labels))

Total images: 35710
Total labels: 35710


In [5]:
train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.20,
    stratify=all_labels,
    random_state=42
)

print("Training:", len(train_files))
print("Temporary:", len(temp_files))

Training: 28568
Temporary: 7142


In [6]:
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train      :", len(train_files))
print("Validation :", len(val_files))
print("Test       :", len(test_files))

Train      : 28568
Validation : 3571
Test       : 3571


In [7]:
def count_classes(labels):
    labels = np.array(labels)

    return {
        "Mask": np.sum(labels == 1),
        "No_mask": np.sum(labels == 0)
    }


print("TRAIN")
print(count_classes(train_labels))

print("\nVALIDATION")
print(count_classes(val_labels))

print("\nTEST")
print(count_classes(test_labels))

TRAIN
{'Mask': np.int64(14558), 'No_mask': np.int64(14010)}

VALIDATION
{'Mask': np.int64(1820), 'No_mask': np.int64(1751)}

TEST
{'Mask': np.int64(1819), 'No_mask': np.int64(1752)}


In [8]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(image, tf.float32) / 255.0

    return image, tf.cast(label, tf.float32)

In [9]:
# Convert file paths to strings
train_paths = [str(p) for p in train_files]
val_paths = [str(p) for p in val_files]
test_paths = [str(p) for p in test_files]

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
)

print("Datasets created successfully!")

Datasets created successfully!


I0000 00:00:1789732716.766798     538 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789732716.769916     538 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [10]:
train_ds = train_ds.map(
    load_image,
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    load_image,
    num_parallel_calls=AUTOTUNE
)

test_ds = test_ds.map(
    load_image,
    num_parallel_calls=AUTOTUNE
)

print("Image preprocessing connected!")

Image preprocessing connected!


In [11]:
train_ds = train_ds.shuffle(
    buffer_size=len(train_paths),
    seed=42
)

train_ds = train_ds.batch(BATCH_SIZE)
val_ds = val_ds.batch(BATCH_SIZE)
test_ds = test_ds.batch(BATCH_SIZE)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("TensorFlow input pipeline ready!")

TensorFlow input pipeline ready!


# ****EXPERIMENT-1 BASELINE MODEL****

In [15]:
def build_custom_cnn():
    
    model = tf.keras.Sequential([
        
        # Input
        tf.keras.layers.Input(shape=(224, 224, 3)),

        # Block 1
        tf.keras.layers.Conv2D(
            32, (3, 3),
            activation="relu",
            padding="same"
        ),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Block 2
        tf.keras.layers.Conv2D(
            64, (3, 3),
            activation="relu",
            padding="same"
        ),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Block 3
        tf.keras.layers.Conv2D(
            128, (3, 3),
            activation="relu",
            padding="same"
        ),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Classification head
        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            128,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    return model


model_e1 = build_custom_cnn()

model_e1.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,938,561 (49.36 MB)

 Trainable params: 12,938,561 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model_e1.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("E1 model compiled successfully!")

E1 model compiled successfully!


In [15]:
EPOCHS = 20

start_time = time.time()

history_e1 = model_e1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

e1_training_time = time.time() - start_time

print(f"\nE1 training time: {e1_training_time / 60:.2f} minutes")

Epoch 1/20
  3/893 ━━━━━━━━━━━━━━━━━━━━ 47s 53ms/step - accuracy: 0.6024 - loss: 1.6922   

I0000 00:00:1789661624.938949     130 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


893/893 ━━━━━━━━━━━━━━━━━━━━ 236s 74ms/step - accuracy: 0.9560 - loss: 0.1252 - val_accuracy: 0.9748 - val_loss: 0.0738
Epoch 2/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 198s 68ms/step - accuracy: 0.9856 - loss: 0.0423 - val_accuracy: 0.9899 - val_loss: 0.0387
Epoch 3/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 180s 72ms/step - accuracy: 0.9914 - loss: 0.0238 - val_accuracy: 0.9874 - val_loss: 0.0483
Epoch 4/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 190s 69ms/step - accuracy: 0.9936 - loss: 0.0189 - val_accuracy: 0.9880 - val_loss: 0.0396
Epoch 5/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 182s 68ms/step - accuracy: 0.9947 - loss: 0.0149 - val_accuracy: 0.9924 - val_loss: 0.0366
Epoch 6/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 187s 68ms/step - accuracy: 0.9966 - loss: 0.0101 - val_accuracy: 0.9796 - val_loss: 0.0840
Epoch 7/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 181s 70ms/step - accuracy: 0.9973 - loss: 0.0097 - val_accuracy: 0.9933 - val_loss: 0.0273
Epoch 8/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 211s 71ms/step - accuracy: 0.9976 - loss: 0.0085 - val

In [16]:
# E1: Test Set Evaluation

test_loss, test_accuracy = model_e1.evaluate(
    test_ds,
    verbose=1
)

print(f"\nE1 Test Loss     : {test_loss:.4f}")
print(f"E1 Test Accuracy : {test_accuracy:.4f}")
print(f"E1 Test Accuracy : {test_accuracy * 100:.2f}%")

112/112 ━━━━━━━━━━━━━━━━━━━━ 14s 126ms/step - accuracy: 0.9776 - loss: 0.2298

E1 Test Loss     : 0.2298
E1 Test Accuracy : 0.9776
E1 Test Accuracy : 97.76%


In [17]:
# Get predictions on the test set

y_true = np.array(test_labels)

y_prob = model_e1.predict(test_ds).ravel()

# Convert probabilities to binary predictions
y_pred = (y_prob >= 0.5).astype(int)

# Calculate metrics
e1_accuracy = accuracy_score(y_true, y_pred)
e1_precision = precision_score(y_true, y_pred)
e1_recall = recall_score(y_true, y_pred)
e1_f1 = f1_score(y_true, y_pred)
e1_roc_auc = roc_auc_score(y_true, y_prob)

print("E1 Results")
print("-" * 30)
print(f"Accuracy  : {e1_accuracy:.4f}")
print(f"Precision : {e1_precision:.4f}")
print(f"Recall    : {e1_recall:.4f}")
print(f"F1-score  : {e1_f1:.4f}")
print(f"ROC-AUC   : {e1_roc_auc:.4f}")

112/112 ━━━━━━━━━━━━━━━━━━━━ 12s 98ms/step
E1 Results
------------------------------
Accuracy  : 0.9776
Precision : 0.9977
Recall    : 0.9582
F1-score  : 0.9776
ROC-AUC   : 0.9989


In [18]:
# E1 Confusion Matrix

cm_e1 = confusion_matrix(y_true, y_pred)

print("E1 Confusion Matrix:")
print(cm_e1)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["No_mask", "Mask"]
    )
)

E1 Confusion Matrix:
[[1748    4]
 [  76 1743]]

Classification Report:
              precision    recall  f1-score   support

     No_mask       0.96      1.00      0.98      1752
        Mask       1.00      0.96      0.98      1819

    accuracy                           0.98      3571
   macro avg       0.98      0.98      0.98      3571
weighted avg       0.98      0.98      0.98      3571



# **E2 - DATA AUGUMENTATION**

In [17]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])



In [18]:
def load_image_aug(path, label):
    image, label = load_image(path, label)
    image = data_augmentation(image, training=True)
    return image, label


train_ds_e2 = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

train_ds_e2 = train_ds_e2.map(
    load_image_aug,
    num_parallel_calls=AUTOTUNE
)

train_ds_e2 = train_ds_e2.shuffle(
    buffer_size=1000,
    seed=42
)

train_ds_e2 = train_ds_e2.batch(32)
train_ds_e2 = train_ds_e2.prefetch(1)

In [19]:
model_e2 = build_custom_cnn()

model_e2.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_e2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,938,561 (49.36 MB)

 Trainable params: 12,938,561 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
EPOCHS = 20

start_time = time.time()

history_e2 = model_e2.fit(
    train_ds_e2,
    validation_data=val_ds,
    epochs=EPOCHS
)

e2_training_time = time.time() - start_time

print(f"\nE2 training time: {e2_training_time / 60:.2f} minutes")

Epoch 1/20
  2/893 ━━━━━━━━━━━━━━━━━━━━ 49s 56ms/step - accuracy: 0.5391 - loss: 1.5335   

I0000 00:00:1789670452.029652    1084 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


893/893 ━━━━━━━━━━━━━━━━━━━━ 316s 336ms/step - accuracy: 0.9409 - loss: 0.1656 - val_accuracy: 0.9667 - val_loss: 0.1045
Epoch 2/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 279s 303ms/step - accuracy: 0.9710 - loss: 0.0839 - val_accuracy: 0.9754 - val_loss: 0.0661
Epoch 3/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 280s 304ms/step - accuracy: 0.9778 - loss: 0.0643 - val_accuracy: 0.9821 - val_loss: 0.0432
Epoch 4/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 277s 301ms/step - accuracy: 0.9820 - loss: 0.0521 - val_accuracy: 0.9866 - val_loss: 0.0356
Epoch 5/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 277s 302ms/step - accuracy: 0.9831 - loss: 0.0490 - val_accuracy: 0.9882 - val_loss: 0.0328
Epoch 6/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 279s 304ms/step - accuracy: 0.9856 - loss: 0.0412 - val_accuracy: 0.9874 - val_loss: 0.0363
Epoch 7/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 275s 300ms/step - accuracy: 0.9882 - loss: 0.0354 - val_accuracy: 0.9905 - val_loss: 0.0240
Epoch 8/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 288s 314ms/step - accuracy: 0.9895 - loss: 0.03

In [21]:
# E2 - Complete Test Evaluation

test_loss, test_accuracy = model_e2.evaluate(
    test_ds,
    verbose=0
)

y_true = np.array(test_labels)
y_prob = model_e2.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

e2_precision = precision_score(y_true, y_pred)
e2_recall = recall_score(y_true, y_pred)
e2_f1 = f1_score(y_true, y_pred)
e2_auc = roc_auc_score(y_true, y_prob)
e2_cm = confusion_matrix(y_true, y_pred)

print("========== E2 RESULTS ==========")
print(f"Accuracy  : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Precision : {e2_precision:.4f}")
print(f"Recall    : {e2_recall:.4f}")
print(f"F1-score  : {e2_f1:.4f}")
print(f"ROC-AUC   : {e2_auc:.4f}")

print("\nConfusion Matrix:")
print(e2_cm)

========== E2 RESULTS ==========
Accuracy  : 0.9924 (99.24%)
Precision : 0.9875
Recall    : 0.9978
F1-score  : 0.9926
ROC-AUC   : 0.9998

Confusion Matrix:
[[1729   23]
 [   4 1815]]


# **E3 - L2 Regularization**

In [13]:
def build_custom_cnn_regularized():
    
    model = tf.keras.Sequential([
        
        tf.keras.layers.Input(shape=(224, 224, 3)),

        # Block 1
        tf.keras.layers.Conv2D(
            32, (3, 3),
            activation="relu",
            padding="same",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Block 2
        tf.keras.layers.Conv2D(
            64, (3, 3),
            activation="relu",
            padding="same",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        tf.keras.layers.MaxPooling2D((2, 2)),

        # Block 3
        tf.keras.layers.Conv2D(
            128, (3, 3),
            activation="relu",
            padding="same",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        tf.keras.layers.MaxPooling2D((2, 2)),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),

        # Dropout
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            1,
            activation="sigmoid"
        )
    ])

    return model


model_e3 = build_custom_cnn_regularized()

model_e3.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,938,561 (49.36 MB)

 Trainable params: 12,938,561 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model_e3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [15]:
EPOCHS = 20

start_time = time.time()

history_e3 = model_e3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

e3_training_time = time.time() - start_time

print(f"\nE3 training time: {e3_training_time / 60:.2f} minutes")

Epoch 1/20
  3/893 ━━━━━━━━━━━━━━━━━━━━ 53s 60ms/step - accuracy: 0.5191 - loss: 1.6377   

I0000 00:00:1789732914.396626     582 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


893/893 ━━━━━━━━━━━━━━━━━━━━ 237s 80ms/step - accuracy: 0.9453 - loss: 0.2026 - val_accuracy: 0.9779 - val_loss: 0.1403
Epoch 2/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 186s 72ms/step - accuracy: 0.9755 - loss: 0.1408 - val_accuracy: 0.9852 - val_loss: 0.1125
Epoch 3/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 188s 72ms/step - accuracy: 0.9836 - loss: 0.1176 - val_accuracy: 0.9868 - val_loss: 0.1036
Epoch 4/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 186s 71ms/step - accuracy: 0.9859 - loss: 0.1084 - val_accuracy: 0.9877 - val_loss: 0.1114
Epoch 5/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 187s 71ms/step - accuracy: 0.9879 - loss: 0.1010 - val_accuracy: 0.9891 - val_loss: 0.0996
Epoch 6/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 185s 72ms/step - accuracy: 0.9900 - loss: 0.0909 - val_accuracy: 0.9908 - val_loss: 0.0910
Epoch 7/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 240s 75ms/step - accuracy: 0.9902 - loss: 0.0904 - val_accuracy: 0.9880 - val_loss: 0.1052
Epoch 8/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 219s 75ms/step - accuracy: 0.9920 - loss: 0.0838 - val

In [16]:
# E3 - Complete Test Evaluation

test_loss, test_accuracy = model_e3.evaluate(
    test_ds,
    verbose=0
)

y_true = np.array(test_labels)

y_prob = model_e3.predict(
    test_ds,
    verbose=0
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

e3_precision = precision_score(y_true, y_pred)
e3_recall = recall_score(y_true, y_pred)
e3_f1 = f1_score(y_true, y_pred)
e3_auc = roc_auc_score(y_true, y_prob)
e3_cm = confusion_matrix(y_true, y_pred)

print("========== E3 RESULTS ==========")
print(f"Accuracy  : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Precision : {e3_precision:.4f}")
print(f"Recall    : {e3_recall:.4f}")
print(f"F1-score  : {e3_f1:.4f}")
print(f"ROC-AUC   : {e3_auc:.4f}")

print("\nConfusion Matrix:")
print(e3_cm)

========== E3 RESULTS ==========
Accuracy  : 0.9941 (99.41%)
Precision : 0.9983
Recall    : 0.9901
F1-score  : 0.9942
ROC-AUC   : 0.9997

Confusion Matrix:
[[1749    3]
 [  18 1801]]


# **E4 - DATA AUGUMENTATION AND L2 REURALIZATION**

In [17]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1)
])

In [18]:
def load_image_e4(path, label):
    image, label = load_image(path, label)
    image = data_augmentation(image, training=True)
    return image, label


train_ds_e4 = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

train_ds_e4 = train_ds_e4.map(
    load_image_e4,
    num_parallel_calls=AUTOTUNE
)

train_ds_e4 = train_ds_e4.shuffle(
    buffer_size=1000,
    seed=42
)

train_ds_e4 = train_ds_e4.batch(32)
train_ds_e4 = train_ds_e4.prefetch(1)

In [19]:
model_e4 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(
        32, 3,
        activation="relu",
        padding="same",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(
        64, 3,
        activation="relu",
        padding="same",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Conv2D(
        128, 3,
        activation="relu",
        padding="same",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),
    tf.keras.layers.MaxPooling2D(2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    ),

    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

model_e4.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,938,561 (49.36 MB)

 Trainable params: 12,938,561 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model_e4.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [21]:
EPOCHS = 20

start_time = time.time()

history_e4 = model_e4.fit(
    train_ds_e4,
    validation_data=val_ds,
    epochs=EPOCHS
)

e4_training_time = time.time() - start_time

print(
    f"\nE4 training time: "
    f"{e4_training_time / 60:.2f} minutes"
)

Epoch 1/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 319s 343ms/step - accuracy: 0.9294 - loss: 0.2516 - val_accuracy: 0.9608 - val_loss: 0.1830
Epoch 2/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 300s 326ms/step - accuracy: 0.9560 - loss: 0.2034 - val_accuracy: 0.9731 - val_loss: 0.1579
Epoch 3/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 297s 323ms/step - accuracy: 0.9644 - loss: 0.1837 - val_accuracy: 0.9672 - val_loss: 0.1618
Epoch 4/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 303s 330ms/step - accuracy: 0.9698 - loss: 0.1660 - val_accuracy: 0.9801 - val_loss: 0.1377
Epoch 5/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 303s 328ms/step - accuracy: 0.9717 - loss: 0.1583 - val_accuracy: 0.9714 - val_loss: 0.1710
Epoch 6/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 301s 328ms/step - accuracy: 0.9760 - loss: 0.1556 - val_accuracy: 0.9888 - val_loss: 0.1161
Epoch 7/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 297s 323ms/step - accuracy: 0.9783 - loss: 0.1409 - val_accuracy: 0.9846 - val_loss: 0.1216
Epoch 8/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 295s 321ms/step - accuracy: 0.9805 -

In [22]:
# E4 - Complete Evaluation

test_loss, test_accuracy = model_e4.evaluate(
    test_ds,
    verbose=0
)

y_true = np.array(test_labels)

y_prob = model_e4.predict(
    test_ds,
    verbose=0
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

e4_precision = precision_score(y_true, y_pred)
e4_recall = recall_score(y_true, y_pred)
e4_f1 = f1_score(y_true, y_pred)
e4_auc = roc_auc_score(y_true, y_prob)
e4_cm = confusion_matrix(y_true, y_pred)

print("========== E4 RESULTS ==========")
print(f"Accuracy  : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Precision : {e4_precision:.4f}")
print(f"Recall    : {e4_recall:.4f}")
print(f"F1-score  : {e4_f1:.4f}")
print(f"ROC-AUC   : {e4_auc:.4f}")

print("\nConfusion Matrix:")
print(e4_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["No_mask", "Mask"]
    )
)

========== E4 RESULTS ==========
Accuracy  : 0.9902 (99.02%)
Precision : 0.9978
Recall    : 0.9830
F1-score  : 0.9903
ROC-AUC   : 0.9994

Confusion Matrix:
[[1748    4]
 [  31 1788]]

Classification Report:
              precision    recall  f1-score   support

     No_mask       0.98      1.00      0.99      1752
        Mask       1.00      0.98      0.99      1819

    accuracy                           0.99      3571
   macro avg       0.99      0.99      0.99      3571
weighted avg       0.99      0.99      0.99      3571

